# Module AD — Orchestrator Agent

**All data and agents below are synthetic/mock demonstration data.** No external service, Google API, MCP capability, model, or raw image is called.

In [ ]:
from dataclasses import dataclass
from spectraderm.agents.orchestrator_agent import OrchestratorAgent, OrchestratorInput

@dataclass(frozen=True)
class SyntheticSafetyResult:
    professional_assessment_recommended: bool

class SyntheticAgent:
    def __init__(self, result):
        self.result, self.calls = result, []
    def interpret(self, value): self.calls.append(value); return self.result
    def evaluate(self, value): self.calls.append(value); return self.result
    def recommend(self, value): self.calls.append(value); return self.result
    def refer(self, value): self.calls.append(value); return self.result

def workflow(referral=False, safety_fails=False):
    vision = SyntheticAgent({'vision': 'structured synthetic output'})
    monitoring = SyntheticAgent({'monitoring': 'structured synthetic output'})
    evidence = SyntheticAgent({'evidence': 'structured synthetic output'})
    safety = SyntheticAgent(SyntheticSafetyResult(referral))
    if safety_fails:
        safety.evaluate = lambda value: (_ for _ in ()).throw(RuntimeError('synthetic safety failure'))
    product = SyntheticAgent({'product': 'synthetic output'})
    referral_agent = SyntheticAgent({'referral': 'synthetic output'})
    result = OrchestratorAgent(vision, monitoring, evidence, safety, product, referral_agent).run(
        OrchestratorInput(vision_input={'input': 'synthetic'}, monitoring_input={'input': 'synthetic'}, referral_location={'city': 'Synthetic City'})
    )
    return result, product.calls, referral_agent.calls


## Case 1 — Product pathway

Vision → Monitoring → Evidence → Safety (`False`) → Product Agent.

In [ ]:
case_1, product_calls, referral_calls = workflow(referral=False)
case_1, len(product_calls), len(referral_calls)


## Case 2 — Referral pathway

Vision → Monitoring → Evidence → Safety (`True`) → Referral Agent. Product Agent is not called.

In [ ]:
case_2, product_calls, referral_calls = workflow(referral=True)
case_2, len(product_calls), len(referral_calls)


## Case 3 — Safety failure/block

A missing/failed Safety result blocks both downstream pathways and produces an explicit blocked result.

In [ ]:
case_3, product_calls, referral_calls = workflow(safety_fails=True)
case_3.status, case_3.errors, product_calls, referral_calls
